# Build Your Own Benchmark: finance MCQ example

This notebook demonstrates the current Nemotron BYOB step structure. It does not import MCQ pipeline internals directly. Instead, it prepares an example corpus, writes a working config from `src/nemotron/steps/byob/config/tiny.yaml`, and runs the public CLI surface: `uv run --no-sync nemotron byob`.

The generated benchmark keeps the MMLU-Pro-style BYOB schema documented in `src/nemotron/steps/byob/references/benchmark-schema.md`.

## What this notebook creates

- `assets/wiki_finance/*.txt`: finance-related source documents.
- `config/finance_wiki.yaml`: a working MCQ config derived from the packaged BYOB tiny config.
- `outputs/finance_wiki_mcq/seed.parquet`: BYOB seed data from the `all` run.
- `outputs/finance_wiki_mcq/benchmark.parquet`: final MCQ benchmark from the `all` run.
- `config/finance_wiki_translate.yaml`: optional translation config derived from the packaged BYOB translation config.

The run cells default to command preview mode so the notebook does not accidentally spend model quota.

## 1. Locate the repository and BYOB step assets

In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
import sys
from pathlib import Path

import yaml


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/nemotron/steps/byob").is_dir():
            return candidate
    raise RuntimeError("Could not find the Nemotron repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
EXAMPLE_DIR = REPO_ROOT / "use-case-examples" / "build-your-own-benchmark"
STEP_CONFIG_DIR = SRC_DIR / "nemotron" / "steps" / "byob" / "config"
STEP_TINY_CONFIG = STEP_CONFIG_DIR / "tiny.yaml"
STEP_TRANSLATE_CONFIG = STEP_CONFIG_DIR / "translate.yaml"

ASSETS_DIR = EXAMPLE_DIR / "assets" / "wiki_finance"
CONFIG_DIR = EXAMPLE_DIR / "config"
OUTPUT_DIR = EXAMPLE_DIR / "outputs"
WORKING_CONFIG = CONFIG_DIR / "finance_wiki.yaml"
TRANSLATION_CONFIG = CONFIG_DIR / "finance_wiki_translate.yaml"

for path in (ASSETS_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository root:", REPO_ROOT)
print("Packaged BYOB tiny config:", STEP_TINY_CONFIG)
print("Example corpus dir:", ASSETS_DIR)
print("Working config:", WORKING_CONFIG)

## 2. Credentials

The BYOB configs use NVIDIA-hosted OpenAI-compatible endpoints. Export `NGC_API_KEY` before starting Jupyter, or set it in this cell for the current notebook process. `NVIDIA_API_KEY` is also accepted. `BYOB_MODEL`, `TRANSLATION_MODEL`, and `TRANSLATION_BASE_URL` can override the hosted model settings. `HF_TOKEN` is optional for public Hugging Face datasets and embedding models.

In [ ]:
NGC_API_KEY = None
NVIDIA_API_KEY = None
BYOB_MODEL = None
TRANSLATION_MODEL = None
TRANSLATION_BASE_URL = None
HF_TOKEN = None

if NGC_API_KEY:
    os.environ["NGC_API_KEY"] = NGC_API_KEY
    os.environ.setdefault("NVIDIA_API_KEY", NGC_API_KEY)

if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
    os.environ.setdefault("NGC_API_KEY", NVIDIA_API_KEY)

if BYOB_MODEL:
    os.environ["BYOB_MODEL"] = BYOB_MODEL

if TRANSLATION_MODEL:
    os.environ["TRANSLATION_MODEL"] = TRANSLATION_MODEL

if TRANSLATION_BASE_URL:
    os.environ["TRANSLATION_BASE_URL"] = TRANSLATION_BASE_URL

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)

print("NGC_API_KEY set:", bool(os.environ.get("NGC_API_KEY")))
print("NVIDIA_API_KEY set:", bool(os.environ.get("NVIDIA_API_KEY")))
print("BYOB_MODEL:", os.environ.get("BYOB_MODEL", "nvidia/nemotron-3-nano-30b-a3b"))
print("TRANSLATION_MODEL:", os.environ.get("TRANSLATION_MODEL", "openai/gpt-oss-120b"))
print("TRANSLATION_BASE_URL:", os.environ.get("TRANSLATION_BASE_URL", "https://integrate.api.nvidia.com/v1"))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")))

## 3. Create a finance source corpus

By default this cell writes small local finance notes so the notebook has a deterministic corpus. To pull live Wikipedia extracts instead, set `DOWNLOAD_FROM_WIKIPEDIA = True`.

In [ ]:
import json
import re
import time
import urllib.parse
import urllib.request

DOWNLOAD_FROM_WIKIPEDIA = False
WIKI_TITLES = ["Finance", "Financial_market", "Bond_(finance)", "Inflation"]


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def fetch_wikipedia_extract(title: str) -> str:
    params = urllib.parse.urlencode(
        {
            "action": "query",
            "format": "json",
            "prop": "extracts",
            "redirects": "1",
            "explaintext": "1",
            "titles": title,
        }
    )
    request = urllib.request.Request(
        f"https://en.wikipedia.org/w/api.php?{params}",
        headers={"User-Agent": "Nemotron-BYOB-example/1.0"},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        data = json.loads(response.read().decode("utf-8"))
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return clean_text(page.get("extract", ""))
    return ""


sample_docs = {
    "finance_overview.txt": "Finance studies how people, firms, and governments allocate money over time. It includes saving, investing, borrowing, lending, budgeting, and risk management. Financial decisions compare expected returns, liquidity needs, and uncertainty.",
    "financial_markets.txt": "Financial markets connect savers and borrowers through instruments such as stocks, bonds, loans, and derivatives. Prices in liquid markets aggregate information about risk, expected cash flows, interest rates, and investor demand.",
    "bonds.txt": "A bond is a debt security. The issuer borrows principal from investors and typically promises periodic coupon payments plus principal repayment at maturity. Bond prices move inversely with market interest rates when other factors are unchanged.",
    "inflation.txt": "Inflation is a sustained increase in the general price level. It reduces purchasing power when wages or investment returns do not keep pace. Central banks often use interest-rate policy to influence inflation and economic activity.",
}

if DOWNLOAD_FROM_WIKIPEDIA:
    for title in WIKI_TITLES:
        text = fetch_wikipedia_extract(title)
        if text:
            (ASSETS_DIR / f"{title}.txt").write_text(f"# {title}\n\n{text}\n", encoding="utf-8")
        time.sleep(0.35)
else:
    for filename, text in sample_docs.items():
        (ASSETS_DIR / filename).write_text(text + "\n", encoding="utf-8")

for path in sorted(ASSETS_DIR.glob("*.txt")):
    print(path.name, path.stat().st_size, "bytes")

## 4. Write the example config from the packaged BYOB config

This keeps the notebook consistent with the framework and the QA pass cases: start from the packaged tiny config, then override paths, run name, model env vars, and small-run controls in an example-local copy.

In [ ]:
cfg = yaml.safe_load(STEP_TINY_CONFIG.read_text(encoding="utf-8"))

cfg.update(
    {
        "expt_name": "finance_wiki_mcq",
        "stage": "all",
        "input_dir": str(EXAMPLE_DIR / "assets"),
        "output_dir": str(OUTPUT_DIR),
        "language": "en-US",
        "source_subjects": ["management", "marketing"],
        "target_source_mapping": {
            "wiki_finance": {
                "subjects": {
                    "management": 0.5,
                    "marketing": 0.5,
                }
            }
        },
        "few_shot_samples_per_query": 1,
        "queries_per_target_subject_document": 1,
        "num_questions_per_query": 1,
        "ndd_batch_size": 2,
        "do_coverage_check": False,
        "remove_hallucinated": False,
        "remove_easy": False,
    }
)
cfg["coverage_check_config"] = {"model_identifier": None, "window_size": None}


def hosted_model(alias: str) -> dict:
    return {
        "alias": alias,
        "model": os.environ.get("BYOB_MODEL", "nvidia/nemotron-3-nano-30b-a3b"),
        "provider": "nvidia",
        "inference_parameters": {
            "max_tokens": 1024,
            "max_parallel_requests": 1,
            "temperature": 0.0,
            "top_p": 1.0,
        },
    }


cfg["generation_model_config"] = hosted_model("finance_generation")
cfg["judge_model_config"] = hosted_model("finance_judge")
cfg["distractor_expansion_model_config"] = hosted_model("finance_distractor_expansion")
cfg["distractor_validity_model_config"] = hosted_model("finance_distractor_validity")
cfg["filtering_model_configs"] = {
    "hallucination": [hosted_model("finance_hallucination")],
    "easiness": [hosted_model("finance_easiness")],
}

WORKING_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(WORKING_CONFIG.read_text(encoding="utf-8"))

## 5. Run BYOB through the Nemotron CLI

These cells expect the repository environment to be synced with `uv sync --extra byob --group run`. They use `uv run --no-sync` so execution does not change the environment mid-notebook.

In [ ]:
BYOB_CLI = ["uv", "run", "--no-sync", "nemotron", "byob"]


def run_nemotron_byob(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_DIR) + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
    cmd = [*BYOB_CLI, *map(str, args)]
    print("$", " ".join(shlex.quote(part) for part in cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, text=True, check=check)


run_nemotron_byob("--list-families")

The next cell defaults to preview mode. Set `RUN_BYOB = True` to run `--stage all`, which chains `prepare` and `generate`. Use separate `prepare`/`generate` stages only when isolating failures.

In [ ]:
RUN_BYOB = False

all_args = ["--family", "mcq", "--stage", "all", "--config", str(WORKING_CONFIG)]

if RUN_BYOB:
    if not (os.environ.get("NGC_API_KEY") or os.environ.get("NVIDIA_API_KEY")):
        raise RuntimeError("NGC_API_KEY or NVIDIA_API_KEY is required before running BYOB generation.")
    run_nemotron_byob(*all_args)
else:
    print("Preview only. Command to run:")
    print(" ".join(shlex.quote(part) for part in [*BYOB_CLI, *all_args]))

## 6. Validate and preview the output parquet

In [ ]:
import pandas as pd
from IPython.display import display

benchmark_dir = OUTPUT_DIR / cfg["expt_name"]
raw_benchmark = benchmark_dir / "benchmark_raw.parquet"
final_benchmark = benchmark_dir / "benchmark.parquet"

print("Raw benchmark:", raw_benchmark, raw_benchmark.exists())
print("Final benchmark:", final_benchmark, final_benchmark.exists())

if final_benchmark.exists():
    df = pd.read_parquet(final_benchmark)
    required_columns = {"question_id", "question", "options", "answer_index", "answer", "src", "category"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise AssertionError(f"Missing required BYOB columns: {sorted(missing_columns)}")
    if df.empty:
        raise AssertionError("BYOB benchmark is empty")
    print({"rows": len(df), "columns": df.columns.tolist()})
    display(df.head())
else:
    print("Run the BYOB stage-all cell first, or point final_benchmark at an existing output.")

## 7. Optional translation config

This mirrors the packaged translation config, points it at the generated `benchmark.parquet`, and keeps hosted model settings in environment variables. Set `RUN_TRANSLATION = True` after the MCQ benchmark exists.

In [ ]:
translate_cfg = yaml.safe_load(STEP_TRANSLATE_CONFIG.read_text(encoding="utf-8"))
translate_cfg.update(
    {
        "expt_name": "finance_wiki_mcq_hi",
        "dataset_path": str(final_benchmark),
        "output_dir": str(OUTPUT_DIR),
        "source_language": "en-US",
        "target_language": "hi-IN",
    }
)
translation_params = translate_cfg["translation_model_config"]["params"]
translation_params.update(
    {
        "alias": "finance_translation",
        "model": os.environ.get("TRANSLATION_MODEL", "openai/gpt-oss-120b"),
        "provider": "nvidia",
        "api_key_env": "NGC_API_KEY",
        "base_url": os.environ.get("TRANSLATION_BASE_URL", "https://integrate.api.nvidia.com/v1"),
    }
)
translation_params.setdefault("inference_parameters", {})["max_parallel_requests"] = 1
TRANSLATION_CONFIG.write_text(yaml.safe_dump(translate_cfg, sort_keys=False), encoding="utf-8")
print(TRANSLATION_CONFIG.read_text(encoding="utf-8"))

RUN_TRANSLATION = False
translation_args = ["--family", "mcq", "--stage", "translate", "--config", str(TRANSLATION_CONFIG)]
translated_benchmark = OUTPUT_DIR / translate_cfg["expt_name"] / "benchmark.parquet"

if RUN_TRANSLATION:
    if not final_benchmark.exists():
        raise RuntimeError("Generate benchmark.parquet before running translation.")
    if not (os.environ.get("NGC_API_KEY") or os.environ.get("NVIDIA_API_KEY")):
        raise RuntimeError("NGC_API_KEY or NVIDIA_API_KEY is required before running BYOB translation.")
    run_nemotron_byob(*translation_args)
    if translated_benchmark.exists():
        translated_df = pd.read_parquet(translated_benchmark)
        missing_columns = required_columns - set(translated_df.columns)
        if missing_columns:
            raise AssertionError(f"Missing required translated BYOB columns: {sorted(missing_columns)}")
        print({"translated_path": str(translated_benchmark), "rows": len(translated_df)})
else:
    print("Preview only. Command to run:")
    print(" ".join(shlex.quote(part) for part in [*BYOB_CLI, *translation_args]))
    print("Translated benchmark target:", translated_benchmark)